# Imbalanced data: when the thing you want is rare

A payments team hunts card fraud among thousands of ordinary transactions. In this notebook you build the same
synthetic fraud data as the lesson, score a model that does nothing, then try class weights, oversampling and a
lower threshold, and finally catch a leak that fakes a perfect score.

About 10 minutes. Run the cells from top to bottom. Everything runs on a free CPU runtime, with no key and no download.

The data is synthetic (made with scikit-learn). It is not real customer data.

## 1. Make the data

The next cell makes 40,000 transactions with 6 number columns each. Only 0.5% of them are fraud.
The seed is fixed, so you get exactly the lesson's data.

**Look for:** 40,000 transactions, 200 of them fraud, a fraud share of 0.005.

In [1]:
import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.utils import resample, compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, confusion_matrix,
                             average_precision_score, roc_auc_score)

SEED = 7
print("scikit-learn version:", sklearn.__version__)

transactions, is_fraud = make_classification(
    n_samples=40_000, n_features=6, n_informative=4, n_redundant=0,
    weights=[0.995], flip_y=0, class_sep=0.9, random_state=SEED)

print("transactions:", len(is_fraud))
print("fraud:", is_fraud.sum())
print("fraud share:", round(is_fraud.mean(), 4))

scikit-learn version: 1.9.1
transactions: 40000
fraud: 200
fraud share: 0.005


## 2. Split it the honest way

`stratify=is_fraud` keeps the fraud share the same in the training part and the test part.
The second half of the cell splits again **without** `stratify`, with ten different seeds, and counts the fraud
that lands in the test part each time.

**Look for:** 30,000 training rows with 150 fraud, 10,000 test rows with 50 fraud.
Without `stratify`, the test fraud count jumps around: 47, 50, 41, 47, 51, 51, 53, 48, 56, 60.

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    transactions, is_fraud, test_size=0.25, stratify=is_fraud, random_state=SEED)
print("train rows:", len(y_train), " train fraud:", y_train.sum())
print("test rows: ", len(y_test), " test fraud: ", y_test.sum())

no_strat = []
for s in range(10):
    _, _, _, yt = train_test_split(transactions, is_fraud, test_size=0.25, random_state=s)
    no_strat.append(int(yt.sum()))
print("test fraud without stratify, seeds 0-9:", no_strat)

train rows: 30000  train fraud: 150
test rows:  10000  test fraud:  50
test fraud without stratify, seeds 0-9: [47, 50, 41, 47, 51, 51, 53, 48, 56, 60]


## 3. The model that always says "not fraud"

`report` prints the confusion matrix and the scores for one model. It is used for every model below.
`DummyClassifier(strategy="most_frequent")` ignores the columns and always gives the most common label.

**Look for:** accuracy 0.995 and recall 0.0. It is right on all 9,950 ordinary transactions and catches none of
the 50 frauds. Average precision is 0.005, the fraud share: that is the do-nothing level.

In [3]:
def report(name, y_true, y_pred, scores=None):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    print(f"--- {name} ---")
    print(f"caught: {tp}  missed: {fn}  false alarms: {fp}  correctly let through: {tn}")
    print("accuracy: ", round(accuracy_score(y_true, y_pred), 4))
    print("precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
    print("recall:   ", round(recall_score(y_true, y_pred), 4))
    if scores is not None:
        print("average precision:", round(average_precision_score(y_true, scores), 4))
        print("ROC AUC:          ", round(roc_auc_score(y_true, scores), 4))

always_no = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
report("always not fraud", y_test, always_no.predict(X_test), always_no.predict_proba(X_test)[:, 1])

--- always not fraud ---
caught: 0  missed: 50  false alarms: 0  correctly let through: 9950
accuracy:  0.995
precision: 0.0
recall:    0.0
average precision: 0.005
ROC AUC:           0.5


## 4. A real model: plain logistic regression

No weights, default threshold 0.5.

**Look for:** accuracy 0.9961, only a hair above 0.995. Recall is 0.24: it catches 12 of 50 and misses 38.
Precision is high (12 of its 13 alarms are real), so it is cautious, not useless.

In [4]:
plain = LogisticRegression(max_iter=1000).fit(X_train, y_train)
plain_scores = plain.predict_proba(X_test)[:, 1]
report("plain logistic regression", y_test, plain.predict(X_test), plain_scores)

--- plain logistic regression ---
caught: 12  missed: 38  false alarms: 1  correctly let through: 9949
accuracy:  0.9961
precision: 0.9231
recall:    0.24
average precision: 0.5378
ROC AUC:           0.8842


## 5. Fix 1: class weights

`compute_class_weight("balanced", ...)` gives each class the weight `n_samples / (2 x rows of that class)`.
Then the model is trained with `class_weight="balanced"`.

**Look for:** weights of about 0.5025 for ordinary and 100.0 for fraud. The weighted model catches 40 of 50,
but it raises 1,332 false alarms to do it. Accuracy falls to 0.8658.

In [5]:
weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
print("weight for ordinary:", round(weights[0], 4), " weight for fraud:", round(weights[1], 4))

weighted = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_train, y_train)
weighted_pred = weighted.predict(X_test)
weighted_scores = weighted.predict_proba(X_test)[:, 1]
print("flagged:", weighted_pred.sum(), "of", len(y_test))
report("weighted logistic regression", y_test, weighted_pred, weighted_scores)

weight for ordinary: 0.5025  weight for fraud: 100.0
flagged: 1372 of 10000
--- weighted logistic regression ---
caught: 40  missed: 10  false alarms: 1332  correctly let through: 8618
accuracy:  0.8658
precision: 0.0292
recall:    0.8
average precision: 0.2527
ROC AUC:           0.9057


## 6. Fix 2: copy the fraud rows (oversampling)

`resample(..., replace=True)` draws fraud rows from the **training part only**, with replacement, until there are
as many fraud rows as ordinary ones.

**Look for:** 59,700 training rows, 29,850 of them fraud, but only 150 different fraud rows. The result is almost
the same as class weights: 41 caught, 1,334 false alarms.

In [6]:
fraud_rows = X_train[y_train == 1]
ok_rows = X_train[y_train == 0]
fraud_up = resample(fraud_rows, replace=True, n_samples=len(ok_rows), random_state=SEED)
X_over = np.vstack([ok_rows, fraud_up])
y_over = np.array([0] * len(ok_rows) + [1] * len(fraud_up))
print("oversampled training rows:", len(y_over))
print("fraud rows:", y_over.sum())
print("different fraud rows:", len(np.unique(fraud_up, axis=0)))

over = LogisticRegression(max_iter=1000).fit(X_over, y_over)
over_scores = over.predict_proba(X_test)[:, 1]
report("oversampled logistic regression", y_test, over.predict(X_test), over_scores)

oversampled training rows: 59700
fraud rows: 29850
different fraud rows: 150
--- oversampled logistic regression ---
caught: 41  missed: 9  false alarms: 1334  correctly let through: 8616
accuracy:  0.8657
precision: 0.0298
recall:    0.82
average precision: 0.249
ROC AUC:           0.9058


## 7. The third lever: a lower threshold

Keep the plain model and say "fraud" at lower and lower scores. The fraud analysts can check at most 300 alerts.

**Look for:** at 0.5 it flags 13 and catches 12. At 0.03 it flags 253 and catches 37, the most within 300 alerts.
At 0.02 it needs 364 alerts. The weighted model catches 40, but with 1,372 alerts.

In [7]:
BUDGET = 300
for threshold in [0.5, 0.2, 0.1, 0.05, 0.03, 0.02, 0.01]:
    flagged = plain_scores >= threshold
    caught = (flagged & (y_test == 1)).sum()
    note = "" if flagged.sum() <= BUDGET else "  (over budget)"
    print(f"threshold {threshold}: {flagged.sum()} alerts, {caught} of {y_test.sum()} caught{note}")

print("weighted model at 0.5:", weighted_pred.sum(), "alerts,",
      ((weighted_pred == 1) & (y_test == 1)).sum(), "caught")

threshold 0.5: 13 alerts, 12 of 50 caught
threshold 0.2: 42 alerts, 25 of 50 caught
threshold 0.1: 81 alerts, 32 of 50 caught
threshold 0.05: 150 alerts, 35 of 50 caught
threshold 0.03: 253 alerts, 37 of 50 caught
threshold 0.02: 364 alerts, 38 of 50 caught  (over budget)
threshold 0.01: 676 alerts, 38 of 50 caught  (over budget)
weighted model at 0.5: 1372 alerts, 40 caught


## 8. What the scores mean

First, a model that scores at random, to see the do-nothing level of each ranking score.
Then the average fraud score of each model, next to the real fraud share of the test week.

**Look for:** the random scorer gets ROC AUC 0.5472 but average precision only 0.0062 (about the fraud share).
The plain model's scores average 0.0049, close to the real 0.005. The weighted and oversampled models average
about 0.22: fine for flagging, wrong as probabilities.

In [8]:
rng = np.random.default_rng(SEED)
random_scores = rng.random(len(y_test))
print("random scorer  average precision:", round(average_precision_score(y_test, random_scores), 4),
      " ROC AUC:", round(roc_auc_score(y_test, random_scores), 4))

print("real fraud share in test:", round(y_test.mean(), 4))
print("mean score, plain:      ", round(plain_scores.mean(), 4))
print("mean score, weighted:   ", round(weighted_scores.mean(), 4))
print("mean score, oversampled:", round(over_scores.mean(), 4))

random scorer  average precision: 0.0062  ROC AUC: 0.5472
real fraud share in test: 0.005
mean score, plain:       0.0049
mean score, weighted:    0.2236
mean score, oversampled: 0.2235


## 9. The same thing in a few lines of scikit-learn

This is the listing from the lesson's library screen, as is: `stratify` in the split, `class_weight` in the
model, and the scores you print.

**Look for:** 50 fraud in test, 1,372 flagged, recall 0.8, precision 0.029, average precision 0.253.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    transactions, is_fraud, test_size=0.25,
    stratify=is_fraud, random_state=7)
model = LogisticRegression(max_iter=1000,
    class_weight="balanced")
model.fit(X_train, y_train)
flagged = model.predict(X_test)
fraud_score = model.predict_proba(X_test)[:, 1]
print("fraud in test:", y_test.sum(), "flagged:", flagged.sum())
print("recall:", round(recall_score(y_test, flagged), 3))
print("precision:", round(precision_score(y_test, flagged), 3))
print("average precision:", round(average_precision_score(y_test, fraud_score), 3))

fraud in test: 50 flagged: 1372
recall: 0.8
precision: 0.029
average precision: 0.253


## 10. Broken on purpose: recall 1.0

A teammate's notebook. It oversamples the **whole** data set first, then splits, then trains a decision tree.
`score` also counts how many fraud rows in the test part also appear in the training part.
Nothing crashes here: the bug is a score that is too good.

**Look for:** 19,900 test rows, 9,992 of them fraud, and all 9,992 also in training. Recall 1.0, precision 0.9979.
A real week is 0.5% fraud, not half.

In [10]:
def oversample(X, y):
    fraud_up = resample(X[y == 1], replace=True, n_samples=int((y == 0).sum()), random_state=SEED)
    return np.vstack([X[y == 0], fraud_up]), np.array([0] * int((y == 0).sum()) + [1] * len(fraud_up))


def score(name, model, X_test, y_test, X_train):
    pred = model.predict(X_test)
    train_set = {r.tobytes() for r in X_train}
    copies = sum(r.tobytes() in train_set for r in X_test[y_test == 1])
    print(f"--- {name} ---")
    print("test rows:", len(y_test), " test fraud:", y_test.sum(), " test fraud also in train:", copies)
    print("recall:   ", round(recall_score(y_test, pred), 4))
    print("precision:", round(precision_score(y_test, pred, zero_division=0), 4))


# BROKEN: oversample everything, then split
X_bal, y_bal = oversample(transactions, is_fraud)
X_train, X_test, y_train, y_test = train_test_split(X_bal, y_bal, test_size=0.25, random_state=SEED)
tree = DecisionTreeClassifier(random_state=SEED).fit(X_train, y_train)
score("broken: oversample, then split", tree, X_test, y_test, X_train)

--- broken: oversample, then split ---
test rows: 19900  test fraud: 9992  test fraud also in train: 9992
recall:    1.0


precision: 0.9979


## 11. The fix: split first, oversample only the training part

**Look for:** 10,000 test rows with 50 fraud, none of them in training. Recall drops to 0.44, precision 0.4783.
That is the honest number.

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    transactions, is_fraud, test_size=0.25, stratify=is_fraud, random_state=SEED)
X_up, y_up = oversample(X_train, y_train)
tree = DecisionTreeClassifier(random_state=SEED).fit(X_up, y_up)
score("fixed: split, then oversample", tree, X_test, y_test, X_up)

--- fixed: split, then oversample ---
test rows: 10000  test fraud: 50  test fraud also in train: 0
recall:    0.44
precision: 0.4783


## Now change it

Three small experiments. Each cell has one line marked `# <- change this`. Edit it, run the cell, and compare
with the numbers above. The cells use `X_train`, `X_test`, `y_train`, `y_test` from step 11, which are the same
honest split as step 2.

### Experiment 1: a milder weight

The cell trains logistic regression with fraud weighted 10 instead of 100. It starts at 10.

**What should change:** weight 10 catches 28 with 37 false alarms: fewer caught than the balanced model (40),
but far fewer false alarms than its 1,332. Weight 1 gives back the plain model (12 caught, 1 false alarm).
As you raise it to 30 and 100, caught fraud and false alarms both go up. The weight only moves the trade.

In [12]:
fraud_weight = 10  # <- change this (try 1, 30, 100)
m = LogisticRegression(max_iter=1000, class_weight={0: 1, 1: fraud_weight}).fit(X_train, y_train)
p = m.predict(X_test)
print("fraud weight:", fraud_weight)
print("caught:", ((p == 1) & (y_test == 1)).sum(), "of", y_test.sum(), " false alarms:", ((p == 1) & (y_test == 0)).sum())

fraud weight: 10
caught: 28 of 50  false alarms: 37


### Experiment 2: your own alert budget

The cell lowers the plain model's threshold until the alerts would go over a budget, and prints the best threshold.

**What should change:** with 300 it picks threshold 0.03 and catches 37. With 100 it catches 32; with 1,000 it
catches 38, and the extra 400 or so alerts buy only one more fraud. The model is the same; only the number you can afford to check changes.

In [13]:
budget = 300  # <- change this (try 100, 1000)
plain = LogisticRegression(max_iter=1000).fit(X_train, y_train)
s = plain.predict_proba(X_test)[:, 1]
best = None
for t in np.round(np.arange(0.5, 0.0, -0.005), 3):
    if (s >= t).sum() > budget:
        break
    best = t
flagged = s >= best
print("budget:", budget, " threshold:", best, " alerts:", flagged.sum(),
      " caught:", (flagged & (y_test == 1)).sum(), "of", y_test.sum())

budget: 300  threshold: 0.03  alerts: 253  caught: 37 of 50


### Experiment 3: make fraud less rare

The cell remakes the data with a different share of ordinary transactions and reruns the do-nothing model and
the plain model. It starts at 0.995, the lesson's data.

**What should change:** set it to 0.9 (10% fraud). The do-nothing accuracy falls to 0.9, so the baseline is
easier to beat, and the plain model's recall rises from 0.24 to 0.66 because it now sees many more fraud rows. Accuracy
was hiding the problem only because fraud was so rare.

In [14]:
ordinary_share = 0.995  # <- change this (try 0.9)
X, y = make_classification(
    n_samples=40_000, n_features=6, n_informative=4, n_redundant=0,
    weights=[ordinary_share], flip_y=0, class_sep=0.9, random_state=SEED)
Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
base = DummyClassifier(strategy="most_frequent").fit(Xa, ya).predict(Xb)
p = LogisticRegression(max_iter=1000).fit(Xa, ya).predict(Xb)
print("fraud share:", round(y.mean(), 4))
print("do-nothing accuracy:", round(accuracy_score(yb, base), 4))
print("plain model accuracy:", round(accuracy_score(yb, p), 4), " recall:", round(recall_score(yb, p), 4))

fraud share: 0.005
do-nothing accuracy: 0.995
plain model accuracy: 0.9961  recall: 0.24


## What you should see

- Step 3: the always-"not fraud" model prints accuracy 0.995 and recall 0.0.
- Step 5: `compute_class_weight("balanced", ...)` gives about 0.5025 for ordinary and 100.0 for fraud.
- Step 5: the weighted model flags 1,372 of 10,000 test transactions and catches 40 of 50.
- Step 6: the oversampled training set has 29,850 fraud rows but only 150 different ones.
- Steps 10 and 11: oversampling before the split reports recall 1.0; splitting first reports 0.44.